In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch
from huggingface_hub import login



/Users/margauxlanglois/Documents/SNCF/s2025p2-mobile-app-incidents/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import json
import os
with open("../../env/keys.json", "r", encoding="utf-8") as f:
    keys = json.load(f)


HF_KEY = keys["HF_TOKEN"]


In [7]:
# Utiliser le modèle CroissantLLM
model_name = "croissantllm/CroissantLLMBase"  # Modèle Croissant LLM

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 29.83it/s]


In [ ]:
with open("../incidents_listes/incidents_categories.json", "r", encoding="utf-8") as f:
    incident_data = json.load(f)

with open("../incidents_listes/incidents_arbo_simple.json", "r", encoding="utf-8") as f:
    incident_arbo = json.load(f)


Cassé, dégradé, manquant


In [14]:
#arbo_str = json.dumps(incident_data, indent=2, ensure_ascii=False)["Localisation (QR)"]

prompt = f"""Tu es un assistant SNCF chargé d’analyser des incidents audio.

Voici l’arborescence complète des incidents, classée par :
- Localisation
- Catégorie
- Objet
- Nature du problème

Lis bien cette structure et mémorise-la, avec les termes exacts. Tu l’utiliseras ensuite pour comprendre des transcriptions et répondre précisément.

Voici la liste des localisations. Je te donnerai le reste ensuite.
{incident_data["Localisation (QR)"]}


Ne fais aucune analyse pour le moment. Dis simplement "Structure comprise." si tu as bien mémorisé l’arborescence.
"""


In [19]:
# On génère le dictionnaire simplifié
give = {k: list(v.keys()) if isinstance(v, dict) else v for k, v in incident_arbo.items()}
print(give)  # Facultatif, pour vérifier

# Construction correcte du prompt avec f-string
prompt = f"""Voici maintenant la liste des catégories d'incidents possibles pour chaque localisation.
{give}

Apprends cette correspondance et dis "Correspondance comprise." si tu as bien mémorisé la liste des catégories d'incidents possibles.
"""



{'Compartiment': ['Accessoires/ Environnement', 'Habillage', 'Pack inoui'], 'Local ASCT': ['Accessoires/ Environnement', 'Eclairage', 'Equipements chauds', 'Equipements froids', 'Equipements SECURITE', 'Habillage', 'Info/Communication', 'Porte local', 'Prise 220 Volts', 'Siège', 'Pack inoui', 'Porte coulissante'], 'Local de service': ['Accessoires/ Environnement', 'Climatisation', 'Eclairage', 'Habillage', 'Info/Communication', 'Porte local', 'Prise 220 Volts', 'Equipements SECURITE'], 'Nurserie': ['Accessoires/ Environnement', 'Climatisation', 'Eclairage', 'Habillage', 'Info/Communication', 'Porte local', 'Prise 220 Volts', 'Lave-mains', 'Pack inoui'], 'Office bar': ['Equipements froids', 'Accessoires/ Environnement', 'Equipements chauds', 'Equipements SECURITE', 'Eclairage', 'Habillage', 'Info/Communication', 'Porte office', 'Pack inoui', 'Prise 220 Volts', 'Four Business'], 'Place': ['Accessoires/ Environnement', 'Equipements SECURITE', 'Eclairage', 'Habillage', 'Info/Communication'

In [20]:
inputs = tokenizer(prompt, return_tensors="pt").to(device=model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)

output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(output.replace(prompt, "").strip())

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


import json
import re

with open('données', 'r', encoding='utf-8') as f:
    data = json.load(f)
    for c in data['locations']:
        if c['location'] in locs:
            print(c['location'])
            print('{}'.format(c['location']))
            locs = c['locations']
            for loc in locs:
                if loc['name'] in locs:
                    print(loc['name'])
                    print('{}'.format(loc['name']))
                    locs = locs[:]
                    break
